# 01 — Preparacao dos Dados

**Objetivo:** Carregar o corpus da Folha de Sao Paulo, binarizar labels (`mercado` vs `outros`), gerar splits 3-way estratificados **80/10/10**, gerar **10 folds de CV estratificada** sobre os 90% (train+val) e persistir tudo em `artifacts/splits/`.

**Mudancas em relacao a versao anterior:**
- Splits agora sao 80/10/10 (eram 64/16/20).
- Adicionado `cv_folds.json` com 10 folds estratificados sobre train+val.
- `balanced_train.parquet` **nao** e mais gerado. O fluxo padrao roda sem balanceamento; a funcao `build_balanced_training_frame` permanece disponivel para reproduzir resultados antigos.
- Coluna multiclasse `label_multi` (top-7 + `outros`) tambem e persistida nos parquets para uso nos notebooks 1x (TF-IDF) e 91 (smoke) sem recalcular.

**Dependencias:** `economy_classifier.datasets`

In [1]:
import hashlib
import json
from pathlib import Path

import pandas as pd

from economy_classifier.datasets import (
    attach_multiclass_label,
    build_cv_folds,
    build_train_val_test_split,
)
from economy_classifier.project import (
    PROJECT_ROOT,
    SPLITS_DIR,
    get_git_commit_short,
    utc_now_iso,
)

In [2]:
CORPUS_PATH = PROJECT_ROOT / "data" / "news-of-the-site-folhauol" / "articles.csv"
SEED = 2026  # global default (was 42 historically; persisted splits in artifacts/splits/ retain seed=42 — re-running this notebook regenerates with seed=2026 and invalidates downstream runs)
POSITIVE_LABEL = "mercado"
TEXT_COLUMN = "text"
LABEL_COLUMN = "label"
TEST_SIZE = 0.10
VAL_SIZE = 0.10
N_CV_FOLDS = 5

## 1. Carga do corpus

In [3]:
raw = pd.read_csv(CORPUS_PATH)
print(f"Corpus carregado: {len(raw):,} artigos")
print(f"Colunas: {list(raw.columns)}")
raw["category"].value_counts().head(15)

Corpus carregado: 167,053 artigos
Colunas: ['title', 'text', 'date', 'category', 'subcategory', 'link']


category
poder             22022
colunas           21622
mercado           20970
esporte           19730
mundo             17130
cotidiano         16967
ilustrada         16345
opiniao            4525
paineldoleitor     4011
saopaulo           3955
tec                2260
tv                 2142
educacao           2118
turismo            1903
ilustrissima       1411
Name: count, dtype: int64

## 2. Binarizacao e limpeza

In [4]:
corpus = raw.copy()
corpus[LABEL_COLUMN] = (corpus["category"] == POSITIVE_LABEL).astype(int)
corpus = corpus.dropna(subset=[TEXT_COLUMN]).reset_index(drop=True)

# Multiclass label (top-7 + outros) attached upfront so 91_smoke_multiclasse_tfidf.ipynb
# does not need to recompute it from raw category.
corpus = attach_multiclass_label(corpus)

n_mercado = int(corpus[LABEL_COLUMN].sum())
n_outros = len(corpus) - n_mercado
print(f"Corpus limpo: {len(corpus):,} artigos")
print(f"  mercado: {n_mercado:,} ({n_mercado/len(corpus)*100:.1f}%)")
print(f"  outros:  {n_outros:,} ({n_outros/len(corpus)*100:.1f}%)")
print("\nlabel_multi distribuicao:")
print(corpus["label_multi"].value_counts())

Corpus limpo: 166,288 artigos
  mercado: 20,970 (12.6%)
  outros:  145,318 (87.4%)

label_multi distribuicao:
label_multi
outros       32233
poder        22022
colunas      21619
mercado      20970
esporte      19730
mundo        17130
cotidiano    16967
ilustrada    15617
Name: count, dtype: int64[pyarrow]


## 3. Splits 3-way estratificados (80/10/10)

Test e validacao preservam a distribuicao natural (~12,5%). Estratificacao por `label` binario.

In [5]:
train, val, test = build_train_val_test_split(
    corpus,
    label_column=LABEL_COLUMN,
    seed=SEED,
    test_size=TEST_SIZE,
    val_size=VAL_SIZE,
)

for name, split in [("Treino", train), ("Validacao", val), ("Teste", test)]:
    pct = split[LABEL_COLUMN].mean() * 100
    print(f"{name:>10}: {len(split):>6,} linhas ({pct:.1f}% mercado)")

print(f"\n  Total: {len(train)+len(val)+len(test):,} == {len(corpus):,}")

    Treino: 133,030 linhas (12.6% mercado)
 Validacao: 16,629 linhas (12.6% mercado)
     Teste: 16,629 linhas (12.6% mercado)

  Total: 166,288 == 166,288


## 4. Cross-validation 10-fold sobre os 90% (train + val)

Os 10 folds sao rotativos sobre o pool train+val unidos (90% do corpus). O test set (10%) permanece intocado em todos os folds. Estratificacao por `label` binario.

In [6]:
train_val_pool = pd.concat([train, val]).sort_index()
print(f"Pool train+val: {len(train_val_pool):,} linhas")

cv_folds = build_cv_folds(
    train_val_pool,
    label_column=LABEL_COLUMN,
    n_folds=N_CV_FOLDS,
    seed=SEED,
)

print(f"Folds gerados: {len(cv_folds)}")
for i, fold in enumerate(cv_folds[:3]):
    n_tr = len(fold["train_indices"])
    n_va = len(fold["val_indices"])
    val_subset = train_val_pool.loc[fold["val_indices"]]
    pct = val_subset[LABEL_COLUMN].mean() * 100
    print(f"  fold {i}: train={n_tr:,}  val={n_va:,}  val mercado={pct:.1f}%")

Pool train+val: 149,659 linhas
Folds gerados: 5
  fold 0: train=119,727  val=29,932  val mercado=12.6%
  fold 1: train=119,727  val=29,932  val mercado=12.6%


  fold 2: train=119,727  val=29,932  val mercado=12.6%


## 5. Persistencia

In [7]:
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

# Indices (CSV) — facilitam reproducao sem carregar parquets
pd.DataFrame({"index": train.index}).to_csv(SPLITS_DIR / "train_indices.csv", index=False)
pd.DataFrame({"index": val.index}).to_csv(SPLITS_DIR / "val_indices.csv", index=False)
pd.DataFrame({"index": test.index}).to_csv(SPLITS_DIR / "test_indices.csv", index=False)

# DataFrames completos (parquet) — incluem text, label e label_multi
train.to_parquet(SPLITS_DIR / "train.parquet")
val.to_parquet(SPLITS_DIR / "val.parquet")
test.to_parquet(SPLITS_DIR / "test.parquet")

# Folds de CV (JSON serializavel)
(SPLITS_DIR / "cv_folds.json").write_text(
    json.dumps(cv_folds, indent=2),
    encoding="utf-8",
)

# Aposentar artefato legado se existir
legacy_balanced = SPLITS_DIR / "balanced_train.parquet"
if legacy_balanced.exists():
    legacy_balanced.unlink()
    print("Removido legado: balanced_train.parquet")

# Hash para rastreabilidade
corpus_hash = hashlib.sha256(
    pd.util.hash_pandas_object(corpus[[TEXT_COLUMN, LABEL_COLUMN]]).values.tobytes()
).hexdigest()

metadata = {
    "seed": SEED,
    "split_scheme": "80/10/10",
    "test_size": TEST_SIZE,
    "val_size": VAL_SIZE,
    "cv_n_folds": N_CV_FOLDS,
    "cv_pool": "train+val (90%)",
    "balanced": False,
    "corpus_sha256": corpus_hash,
    "corpus_rows": len(corpus),
    "train_rows": len(train),
    "val_rows": len(val),
    "test_rows": len(test),
    "train_mercado_pct": round(float(train[LABEL_COLUMN].mean() * 100), 2),
    "val_mercado_pct": round(float(val[LABEL_COLUMN].mean() * 100), 2),
    "test_mercado_pct": round(float(test[LABEL_COLUMN].mean() * 100), 2),
    "label_multi_top_classes": list(corpus["label_multi"].value_counts().head(8).index),
    "generated_at": utc_now_iso(),
    "git_commit": get_git_commit_short(),
}

(SPLITS_DIR / "split_metadata.json").write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("Splits persistidos em", SPLITS_DIR)
print(json.dumps(metadata, indent=2, ensure_ascii=False))

Splits persistidos em /home/diacrono/Documentos/repositorios/economy-classifier/artifacts/splits
{
  "seed": 42,
  "split_scheme": "80/10/10",
  "test_size": 0.1,
  "val_size": 0.1,
  "cv_n_folds": 5,
  "cv_pool": "train+val (90%)",
  "balanced": false,
  "corpus_sha256": "e397609bd226ec7f5189efac26c6045c59adda50a82f06c2a2733d1ee95c787d",
  "corpus_rows": 166288,
  "train_rows": 133030,
  "val_rows": 16629,
  "test_rows": 16629,
  "train_mercado_pct": 12.61,
  "val_mercado_pct": 12.61,
  "test_mercado_pct": 12.61,
  "label_multi_top_classes": [
    "outros",
    "poder",
    "colunas",
    "mercado",
    "esporte",
    "mundo",
    "cotidiano",
    "ilustrada"
  ],
  "generated_at": "2026-04-26T14:54:42+00:00",
  "git_commit": "14f7cff"
}
